# ETF Arbitrage Signal Detection

### Market Trading Algorithmic Signals — Banking-AI

This notebook explains how traders find profit opportunities by spotting price differences between an ETF and its component stocks:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), and basic understanding of market trading and algorithmic signals terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe the information-extraction task and its relevance to market trading and algorithmic signals.
- Implement a rule-based extraction pipeline and evaluate accuracy on synthetic examples.
- Assess limitations of rule-based extraction and when ML-based approaches would be more appropriate.

**Estimated time:** 35–45 minutes

**Collection context:** Volatility forecasting, trading signals, price prediction, and quantitative market analysis.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** An Exchange-Traded Fund (ETF) is just a basket of stocks. In a perfect world, the price of the ETF should exactly match the total price of the stocks inside it. However, due to market delays, the ETF price often drifts away from its "Fair Value". High-frequency traders use algorithms to spot these gaps and trade them, pushing the prices back together. This is called **Arbitrage**.

**Input data used:** ETF Price and Basket Fair Value (updated every minute).

**What we predict:** Arbitrage signals (Buy Basket/Sell ETF or vice versa).

**ML method used:** Statistical Signal Processing (Z-Score Thresholding).

**Learning goal:** Understand how to detect anomalies in price relationships.

**Primary stakeholders:** quantitative analysts, portfolio managers, risk officers, and trading desk supervisors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic ETF & Basket Data

We simulate minute-by-minute data where the ETF price follows the Basket price with some noise and occasional "jumps".

In [ ]:
n_minutes = 1440 # One full trading day (approx)
rng = RNG

# 1. Simulate the Basket "Fair Value" (Random Walk)
basket_returns = rng.normal(0, 0.0002, n_minutes)
basket_price = 100 * np.exp(np.cumsum(basket_returns))

# 2. Simulate the ETF Price (Follows basket + small noise + arbitrage gaps)
etf_price = basket_price.copy() + rng.normal(0, 0.01, n_minutes)

# Add artificial arbitrage opportunities (ETF drifts away)
etf_price[400:450] += 0.15 # ETF becomes overvalued
etf_price[900:950] -= 0.15 # ETF becomes undervalued

df = pd.DataFrame({
    'minute': range(n_minutes),
    'basket_price': basket_price,
    'etf_price': etf_price
})

plt.figure(figsize=(12, 5))
plt.plot(df['basket_price'], label='Basket Fair Value', color='gray', alpha=0.8)
plt.plot(df['etf_price'], label='ETF Market Price', color='#2A9D8F', alpha=0.6)
plt.title('ETF Price vs. Underlying Basket Fair Value')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

## Step 4 — Exploratory Data Analysis

Calculate the Basis and Z-Score

The **Basis** is simply `ETF_Price - Basket_Price`. We then normalize it into a **Z-Score** to see how many standard deviations the gap is from the average.

In [ ]:
df['basis'] = df['etf_price'] - df['basket_price']

# Calculate rolling Z-Score
window = 60
df['mean_basis'] = df['basis'].rolling(window).mean()
df['std_basis'] = df['basis'].rolling(window).std()
df['z_score'] = (df['basis'] - df['mean_basis']) / df['std_basis']

plt.figure(figsize=(12, 4))
plt.plot(df['z_score'], color='purple')
plt.axhline(2, color='#E76F51', linestyle='--', label='Sell ETF (Overvalued)')
plt.axhline(-2, color='#264653', linestyle='--', label='Buy ETF (Undervalued)')
plt.title('Arbitrage Z-Score Signal')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Line chart titled "Arbitrage Z-Score Signal". The chart shows temporal trends and the alignment between predicted and observed values.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

For this workflow, the data prepared in Step 3 is used directly as input features.

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: manual extraction (no automation) ---
print("Baseline: without automation, each document must be read and keyed manually.")
print("Any automated approach that achieves >80% field accuracy saves significant time.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
df['signal'] = 0
df.loc[df['z_score'] > 2, 'signal'] = -1 # Sell Signal
df.loc[df['z_score'] < -2, 'signal'] = 1 # Buy Signal

print(f"Number of Buy Signals: {len(df[df['signal']==1])}")
print(f"Number of Sell Signals: {len(df[df['signal']==-1])}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df['etf_price'], label='ETF Price', color='#2A9D8F', alpha=0.3)

# Plot Buy signals
plt.scatter(df[df['signal'] == 1].index, df[df['signal'] == 1]['etf_price'], 
            color='#264653', marker='^', label='Buy Signal', alpha=1)

# Plot Sell signals
plt.scatter(df[df['signal'] == -1].index, df[df['signal'] == -1]['etf_price'], 
            color='#E76F51', marker='v', label='Sell Signal', alpha=1)

plt.title('Arbitrage Signals on ETF Price Chart')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Scatter plot, line chart titled "Arbitrage Signals on ETF Price Chart". The chart highlights spatial separation or clustering patterns in the data.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- We successfully identified periods where the ETF price deviated from its fair value.
- The Z-score helped us ignore small random noise and only trade when the gap was statistically significant.

**Market Context:** In the real world, these gaps disappear in milliseconds as thousands of algorithms compete to trade them. Banks provide the "liquidity" for these ETFs by ensuring they stay close to fair value, profiting from these tiny price discrepancies.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: process a new document ---
print("Operational demonstration — field extraction:\n")
new_doc = "Invoice #2055. Vendor: Wayne Enterprises. Date: 2024-06-15. Total: $8750.00"
result = extract_info(new_doc)
print(f"  Raw: {new_doc}")
print(f"  Vendor: {result[0]}  Date: {result[1]}  Amount: ${result[2]:,.2f}")
print("\nExtracted fields would auto-populate the accounts-payable system.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end market trading and algorithmic signals workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Algorithmic trading models can amplify market volatility and systemic risk if deployed without safeguards.
- Backtested performance often overstates real-world returns due to look-ahead bias and transaction costs.
- Automated trading decisions must include human oversight and circuit breakers.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in market trading and algorithmic signals?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.